# Module B - radiogenomics and pseudoprogression

Two independent analyses that share code but never patients:

1. **UPENN-GBM** - MGMT promoter methylation from baseline pre-operative MRI.
2. **UCSD-PTGBM** - tumour vs treatment-related change at a post-treatment
   timepoint.

Run the CLI first:

```
gbmplast radiogenomics --dataset upenn_gbm --target mgmt
gbmplast radiogenomics --dataset ucsd_ptgbm
```

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from gbmplast.config import load_config
from gbmplast.report.theme import apply_theme, series_color

apply_theme()
config = load_config()
results = config.paths.results


def load(dataset: str, analysis: str, name: str) -> pd.DataFrame | None:
    path = results / dataset / analysis / f"{name}.csv"
    return pd.read_csv(path) if path.exists() else None

## UPENN-GBM: MGMT from pre-operative MRI

The estimand is declared before any model is fitted, and travels with the
results.

In [2]:
estimand = load("upenn_gbm", "radiogenomics_mgmt", "estimand")
if estimand is not None:
    for _, row in estimand.iterrows():
        print(f"{row['field']:28s} {row['value']}")

name                         upenn_mgmt_from_baseline_mri
question                     Can MGMT promoter methylation status be predicted from pre-operative MRI radiomics and routinely available clinical covariates?
cohort                       UPENN-GBM
unit_of_analysis             baseline pre-operative scan (one per patient)
outcome                      mgmt_label
outcome_type                 binary
prediction_time              the baseline pre-operative scan, before surgery
lookback_window_days         0
outcome_horizon_days         nan
eligibility                  scan is the baseline pre-operative study (suffix _11); MGMT promoter methylation is reported as a definite call; CaPTk radiomic features are available for the scan
exclusions                   recurrence scans (suffix _21): a post-treatment scan is not a baseline, and mixing the two would put two scans of one patient in the analysis; Indeterminate / NOS-NEC / Not Available marker calls
positive_label               methyla

In [3]:
load("upenn_gbm", "radiogenomics_mgmt", "cohort_flow")

,kind,label,n
0,step,scans in cohort,671
1,step,scans with a definite MGMT call,291
2,step,restricted to baseline pre-operative scans,262
3,step,scans with CaPTk radiomics,611
4,step,analysis set,262
5,step,patients,262
6,step,positive (methylated),111


### Performance

All three feature blocks sit at chance. Under patient-grouped splitting with
feature selection inside the fold, pre-operative radiomics do not predict
MGMT status in this cohort. Published claims of high AUC for this task
frequently come from row-level splits or selection performed before
splitting; neither is done here.

In [4]:
metrics = load("upenn_gbm", "radiogenomics_mgmt", "model_metrics")
if metrics is not None:
    keep = [c for c in ("model", "n_rows", "n_patients", "prevalence",
                        "auroc", "auprc", "brier", "calibration_slope")
            if c in metrics.columns]
    display = metrics[keep].round(4)
    print(display.to_string(index=False))

                           model  n_rows  n_patients  prevalence  auroc  auprc  brier  calibration_slope
             prevalence_baseline     262         262      0.4237 0.4175 0.3815 0.2474            -3.9977
      clinical_only__elastic_net     262         262      0.4237 0.5350 0.4585 0.2513             0.3413
clinical_only__gradient_boosting     262         262      0.4237 0.4543 0.4076 0.2873            -0.1199
           mri_only__elastic_net     262         262      0.4237 0.4588 0.3979 0.2926            -0.1638
     mri_only__gradient_boosting     262         262      0.4237 0.4519 0.3896 0.3108            -0.2016
           combined__elastic_net     262         262      0.4237 0.4588 0.3979 0.2926            -0.1638
     combined__gradient_boosting     262         262      0.4237 0.4490 0.3885 0.3170            -0.1778


In [5]:
ci = load("upenn_gbm", "radiogenomics_mgmt", "model_metric_ci")
if ci is not None:
    auroc = ci[ci.metric == "auroc"]
    fig, ax = plt.subplots(figsize=(7, 0.42 * len(auroc) + 1.4))
    y = np.arange(len(auroc))
    ax.axvline(0.5, color="#b9b8b1", linestyle=(0, (4, 3)), linewidth=1.2)
    for i, (_, row) in enumerate(auroc.iterrows()):
        ax.plot([row.ci_low, row.ci_high], [i, i], color=series_color(0),
                linewidth=2)
        ax.scatter([row.estimate], [i], s=42, color=series_color(0),
                   edgecolor="white", zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels([m.replace("__", " + ") for m in auroc.model],
                       fontsize=8)
    ax.set_xlabel("AUROC (patient bootstrap 95% CI)")
    ax.set_xlim(0.3, 1.0)
    ax.set_title("MGMT from baseline MRI - every block includes 0.5")
    ax.grid(axis="y", visible=False)
    plt.show()

## UCSD-PTGBM: tumour vs treatment-related change

The cohort's distinguishing feature is that it records **how** each negative
judgement was made.

In [6]:
load("ucsd_ptgbm", "radiogenomics_tumor_vs_treatment", "label_adjudication")

,outcome_label,outcome_detail,ascertainment,n
0,treatment_related_change,nonspecific_treatment_change,radiologic_followup,29
1,treatment_related_change,pseudoprogression,clinical_consensus,9
2,treatment_related_change,pseudoprogression,histopathology,6
3,treatment_related_change,radiation_necrosis,unknown,7
4,tumor_positive,residual_or_recurrent_tumor,unknown,192


The released tables do not flag which tumour-positive timepoints are
biopsy-proven, so a histopathology-restricted comparison is possible on the
negative class only. The sensitivity analysis says so explicitly rather than
producing a number that would imply otherwise.

In [7]:
load("ucsd_ptgbm", "radiogenomics_tumor_vs_treatment",
     "adjudication_sensitivity")

,sensitivity,n,n_negative,status,reason,model,n_patients,auroc,auprc,prevalence
0,histopathology_only_negatives,198,6,skipped,only 6 negatives after restriction; too few to...,NaN,NaN,NaN,NaN,NaN
1,pathology_or_clinical_negatives,207,15,fitted,NaN,prevalence_baseline,159.0,0.353472,0.904134,0.927536
2,pathology_or_clinical_negatives,207,15,fitted,NaN,combined__elastic_net,159.0,0.519444,0.917520,0.927536
3,pathology_or_clinical_negatives,207,15,fitted,NaN,combined__gradient_boosting,159.0,0.459375,0.914327,0.927536


In [8]:
metrics = load("ucsd_ptgbm", "radiogenomics_tumor_vs_treatment",
               "model_metrics")
if metrics is not None:
    keep = [c for c in ("model", "n_rows", "n_patients", "prevalence",
                        "auroc", "auprc", "brier", "calibration_slope")
            if c in metrics.columns]
    print(metrics[keep].round(4).to_string(index=False))

                                   model  n_rows  n_patients  prevalence  auroc  auprc  brier  calibration_slope
                     prevalence_baseline     243         178      0.7901 0.4390 0.7676 0.1666            -4.3793
              clinical_only__elastic_net     243         178      0.7901 0.6584 0.8624 0.2360             0.4836
        clinical_only__gradient_boosting     243         178      0.7901 0.5917 0.8370 0.1843             0.2419
      treatment_timing_only__elastic_net     243         178      0.7901 0.6094 0.8347 0.2384             0.5187
treatment_timing_only__gradient_boosting     243         178      0.7901 0.5266 0.8109 0.2129            -0.0101
                   combined__elastic_net     243         178      0.7901 0.5804 0.8101 0.2536             0.1431
             combined__gradient_boosting     243         178      0.7901 0.5852 0.8365 0.2121             0.1178


Note the prevalence beside AUPRC: the positive class is 79%, so an AUPRC of
0.86 is only modestly above the floor. The cohort was deliberately enriched
for pathology-proven cases, so this is **not** the prevalence of tumour in
routine post-treatment imaging.

In [9]:
importance = None
directory = results / "ucsd_ptgbm" / "radiogenomics_tumor_vs_treatment"
for path in directory.glob("permutation_importance__*.csv"):
    importance = pd.read_csv(path)
if importance is not None:
    print(importance.round(4).to_string(index=False))
    print("\nOnly features with sign_stable = True held a consistent "
          "direction across folds.")

                feature  mean_importance  sd_across_folds  min_importance  max_importance  n_folds  sign_stable scoring
        mgmt_methylated           0.0499           0.0559         -0.0192          0.1164        5        False roc_auc
    timepoint_index_num           0.0255           0.0283         -0.0196          0.0480        5        False roc_auc
                    gtr           0.0252           0.0434         -0.0385          0.0805        5        False roc_auc
             idh_mutant           0.0189           0.0192          0.0095          0.0532        5         True roc_auc
       had_radiotherapy           0.0168           0.0366         -0.0368          0.0642        5        False roc_auc
             sex_female           0.0164           0.0192         -0.0116          0.0363        5        False roc_auc
        had_bevacizumab           0.0001           0.0093         -0.0102          0.0152        5        False roc_auc
age_at_first_scan_years          -0.0007

## Why the imaging arm is missing here

UCSD-PTGBM distributes its 33 GB NIfTI bundle through IBM Aspera, which is
not usable from a headless environment. The join is implemented: dropping an
`imaging_features.csv` keyed by `local_visit_id` into the cohort's raw
directory activates an imaging block with no change to model code.

In [10]:
from gbmplast.adapters import get_adapter

adapter = get_adapter("ucsd_ptgbm", config)
print("expected imaging feature table:",
      adapter.raw_dir / "imaging_features.csv")
print("present:", (adapter.raw_dir / "imaging_features.csv").exists())

expected imaging feature table: /home/user/gbm_plasticity/data/raw/ucsd_ptgbm/imaging_features.csv
present: False
